In [1]:
import threading
import time
import random
from typing import List

class BackendServer:
    def __init__(self, server_id: str):
        self.server_id = server_id
        self.healthy = True
        self.active_connections = 0
        self.total_served = 0
        self.lock = threading.Lock()

    def handle_request(self):
        with self.lock:
            self.active_connections += 1

        time.sleep(random.uniform(0.1, 0.4))

        with self.lock:
            self.active_connections -= 1
            self.total_served += 1

class LoadBalancer:
    def __init__(self, servers: List[BackendServer], strategy: str = "least_connections"):
        self.servers = servers
        self.strategy = strategy
        self.index = 0
        self.lock = threading.Lock()
        self.running = True

        self.health_thread = threading.Thread(target=self._health_check_daemon, daemon=True)
        self.health_thread.start()

    def _health_check_daemon(self):
        while self.running:
            time.sleep(1.5)
            server = random.choice(self.servers)
            with server.lock:
                server.healthy = not server.healthy
            state = "CRASHED" if not server.healthy else "RECOVERED"
            print(f"\n🚨 [HEALTH MONITOR] {server.server_id} has {state}!\n")

    def get_healthy_servers(self) -> List[BackendServer]:
        return [s for s in self.servers if s.healthy]

    def route_request(self) -> BackendServer:
        healthy_servers = self.get_healthy_servers()
        if not healthy_servers:
            return None

        with self.lock:
            if self.strategy == "round_robin":
                server = healthy_servers[self.index % len(healthy_servers)]
                self.index += 1
                return server

            elif self.strategy == "least_connections":
                return min(healthy_servers, key=lambda s: s.active_connections)

    def shutdown(self):
        self.running = False
        self.health_thread.join()

def client_traffic(lb: LoadBalancer, request_id: int):
    server = lb.route_request()
    if server:
        print(f"[Req {request_id:02d}] Routed to -> {server.server_id} (Active Conns: {server.active_connections})")
        server.handle_request()
    else:
        print(f"[Req {request_id:02d}] 503 Service Unavailable: No healthy backend servers.")

if __name__ == "__main__":
    nodes = [BackendServer(f"Web-Node-{i}") for i in range(1, 4)]
    router = LoadBalancer(nodes, strategy="least_connections")

    print("🚀 Booting L7 Load Balancer (Strategy: Least Connections)...\n")

    threads = []
    for i in range(1, 26):
        t = threading.Thread(target=client_traffic, args=(router, i))
        threads.append(t)
        t.start()
        time.sleep(random.uniform(0.05, 0.15))

    for t in threads:
        t.join()

    router.shutdown()

    print("\n📊 --- TRAFFIC DISTRIBUTION METRICS ---")
    for s in nodes:
        status = "Healthy" if s.healthy else "Offline"
        print(f"{s.server_id} [{status}]: {s.total_served} requests served.")

🚀 Booting L7 Load Balancer (Strategy: Least Connections)...

[Req 01] Routed to -> Web-Node-1 (Active Conns: 0)
[Req 02] Routed to -> Web-Node-2 (Active Conns: 0)
[Req 03] Routed to -> Web-Node-3 (Active Conns: 0)
[Req 04] Routed to -> Web-Node-2 (Active Conns: 0)
[Req 05] Routed to -> Web-Node-1 (Active Conns: 0)
[Req 06] Routed to -> Web-Node-3 (Active Conns: 0)
[Req 07] Routed to -> Web-Node-2 (Active Conns: 0)
[Req 08] Routed to -> Web-Node-1 (Active Conns: 0)
[Req 09] Routed to -> Web-Node-3 (Active Conns: 0)
[Req 10] Routed to -> Web-Node-1 (Active Conns: 1)
[Req 11] Routed to -> Web-Node-2 (Active Conns: 0)
[Req 12] Routed to -> Web-Node-1 (Active Conns: 0)
[Req 13] Routed to -> Web-Node-3 (Active Conns: 0)
[Req 14] Routed to -> Web-Node-2 (Active Conns: 0)

🚨 [HEALTH MONITOR] Web-Node-2 has CRASHED!

[Req 15] Routed to -> Web-Node-1 (Active Conns: 0)
[Req 16] Routed to -> Web-Node-3 (Active Conns: 0)
[Req 17] Routed to -> Web-Node-1 (Active Conns: 0)
[Req 18] Routed to -> Web-N